# ITI113 - SageMaker Pipeline with Post-Pipeline SageMaker MLflow App Logging

This notebook consumes `team07_best_model.json` from Baseline Experiments and operationalises the best leakage-safe Logistic Regression or XGBoost run as a **staging candidate**.
Pipeline controls:
- Pipeline creation and execution are allowed even when the Baseline Experiments quality gate fails
- The best run is always registered with SageMaker status `PendingManualApproval`
- The package is tagged with lifecycle stage `Staging` and its actual `DeploymentReady` result
- Recall and FPR results remain visible for governance and presentation
- Failed candidates cannot be automatically approved or deployed to an endpoint
- `Risk_Score` remains blocked from training and inference
- Raw JSON inference code is prepared but is not invoked until a separately approved deployment exists

In [1]:
# Restart the kernel after this cell if packages were upgraded.
%pip install --quiet --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow

Note: you may need to restart the kernel to use updated packages.


## 1. Initialize Pipeline Configuration and Validate the Leakage-Safe Champion

In [2]:
import base64
import json
import time
from pathlib import Path

import boto3
import sagemaker
from botocore.exceptions import ClientError

REGION = "ap-southeast-1"
COURSE = "ITI113"
SEMESTER = "26S1"
TEAM_ID = "team07"
STUDENT_ID = "s701"
PROJECT_NAME = "credit-card-fraud-detection"
RAW_DATASET_VERSION = "fraud_dataset_50k_v1"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/{PROJECT_NAME}"
RAW_DATA_LOCAL = Path("data/fraud_dataset.csv")
RAW_DATA_S3_URI = f"s3://{BUCKET}/{PREFIX}/data/raw/fraud_dataset.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline/{RAW_DATASET_VERSION}"
SOURCE_DIR = Path("pipeline_src")

PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE = "ml.m5.large"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
boto_session = boto3.Session(region_name=REGION)
sm = boto_session.client("sagemaker")
s3 = boto_session.client("s3")

mlflow_candidates = [
    Path(f"mlflow_app_config_{TEAM_ID}.json"),
    Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json"),
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]
mlflow_config = {}
mlflow_config_used = None
for candidate in mlflow_candidates:
    if candidate.exists():
        mlflow_config = json.loads(candidate.read_text(encoding="utf-8"))
        mlflow_config_used = candidate
        break
if not mlflow_config:
    raise FileNotFoundError("Run Notebook 01 and copy its MLflow configuration file here.")

MLFLOW_APP_ARN = mlflow_config.get("MLFLOW_APP_ARN")
MLFLOW_EXPERIMENT_NAME = mlflow_config.get("EXPERIMENT_NAME", f"{COURSE}/{TEAM_ID}/Experiment1")

CHAMPION_FILE = Path(f"{TEAM_ID}_best_model.json")
if not CHAMPION_FILE.exists():
    raise FileNotFoundError(f"Missing {CHAMPION_FILE}. Run updated Notebook 02 first.")
champion = json.loads(CHAMPION_FILE.read_text(encoding="utf-8"))

if champion.get("team_id") != TEAM_ID:
    raise ValueError("Champion belongs to another team.")
if champion.get("raw_dataset_version") != RAW_DATASET_VERSION:
    raise ValueError("Champion was not trained on the approved 50K dataset version.")
if champion.get("algorithm") not in {"LogisticRegression", "XGBoost"}:
    raise ValueError("Only Model A Logistic Regression or Model B XGBoost can be deployed.")
if champion.get("risk_score_excluded") is not True:
    raise RuntimeError("Risk_Score-inclusive models are blocked from registration and deployment.")
if champion.get("split_strategy") != "stratified_random_60_20_20":
    raise ValueError("Champion split strategy does not match this pipeline.")
SOURCE_DEPLOYMENT_READY = bool(champion.get("deployment_ready", False))
# Governance controls: this notebook may build/register a staging candidate, but
# it never approves a production package or creates an endpoint automatically.
ALLOW_PRODUCTION_APPROVAL = False
ALLOW_ENDPOINT_DEPLOYMENT = False

CHAMPION_ALGORITHM = champion["algorithm"]
CHAMPION_MODEL_FAMILY = champion["model_family"]
CHAMPION_PARAMS = champion["best_parameters"]
DECISION_THRESHOLD = float(champion["decision_threshold"])
MAX_FALSE_POSITIVE_RATE = float(champion["maximum_false_positive_rate"])
MIN_TEST_RECALL = float(
    champion.get("minimum_acceptable_recall", champion.get("minimum_test_recall", 0.60))
)
RANDOM_STATE = int(champion["random_state"])
RAW_INFERENCE_COLUMNS = champion["raw_inference_columns"]
MODEL_FEATURE_COLUMNS = champion["model_feature_columns"]
CATEGORICAL_COLUMNS = champion["categorical_columns"]
NUMERIC_COLUMNS = champion["numeric_columns"]

def encode_json(value):
    return base64.urlsafe_b64encode(json.dumps(value).encode("utf-8")).decode("ascii")

MODEL_PARAMS_B64 = encode_json(CHAMPION_PARAMS)
SCHEMA_B64 = encode_json({
    "raw_inference_columns": RAW_INFERENCE_COLUMNS,
    "model_feature_columns": MODEL_FEATURE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
})

PIPELINE_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-{PROJECT_NAME}"
ENDPOINT_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}"

print("Champion:", CHAMPION_MODEL_FAMILY, "/", CHAMPION_ALGORITHM)
print("Source MLflow run:", champion["best_run_id"])
print("Dataset:", RAW_DATASET_VERSION)
print("Risk_Score excluded:", champion["risk_score_excluded"])
print("Threshold:", DECISION_THRESHOLD)
print("Notebook 02 deployment ready:", SOURCE_DEPLOYMENT_READY)
print("Registration policy: PendingManualApproval / Staging")
print("Automatic production approval:", ALLOW_PRODUCTION_APPROVAL)
print("Automatic endpoint deployment:", ALLOW_ENDPOINT_DEPLOYMENT)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Champion: Model_B / XGBoost
Source MLflow run: 21825ef39b7c41bdadc838f378ae6c2f
Dataset: fraud_dataset_50k_v1
Risk_Score excluded: True
Threshold: 0.19469599425792694
Notebook 02 deployment ready: True
Registration policy: PendingManualApproval / Staging
Automatic production approval: False
Automatic endpoint deployment: False


## 2. Verify Team MLflow Access and Experiment Tracking

In [3]:
import mlflow

app_tags = {item["Key"]: item["Value"] for item in sm.list_tags(ResourceArn=MLFLOW_APP_ARN).get("Tags", [])}
if app_tags.get("TeamId") != TEAM_ID:
    raise PermissionError("MLflow App TeamId does not match this notebook.")

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_pipeline_precheck_{int(time.time())}"):
    mlflow.set_tags({
        "course": COURSE, "semester": SEMESTER, "team_id": TEAM_ID,
        "student_id": STUDENT_ID, "project_name": PROJECT_NAME,
        "dataset_version": RAW_DATASET_VERSION, "run_type": "pipeline_precheck",
    })
    mlflow.log_metric("connection_success", 1)
print("MLflow precheck succeeded.")

🏃 View run team07_s701_pipeline_precheck_1787679615 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/feab68c483464e14b195a51976b9301a
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
MLflow precheck succeeded.


## 3. Generate Leakage-Safe Processing, Training and Inference Scripts

In [4]:
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

preprocess_source = r'''
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


TARGET = "Fraud_Label"
IDENTIFIERS = ["Transaction_ID", "User_ID"]
LEAKAGE_EXCLUDED = ["Risk_Score"]
SUPPLIED_QUALITY_EXCLUDED = ["Is_Weekend"]


def prepare_features(frame):
    result = frame.copy()
    timestamp = pd.to_datetime(result["Timestamp"], errors="raise")
    result["Transaction_Hour"] = timestamp.dt.hour
    result["Transaction_DayOfWeek"] = timestamp.dt.dayofweek
    result["Transaction_Month"] = timestamp.dt.month
    result["Derived_Is_Weekend"] = (timestamp.dt.dayofweek >= 5).astype(int)
    result["Transaction_Hour_Sin"] = np.sin(2 * np.pi * result["Transaction_Hour"] / 24)
    result["Transaction_Hour_Cos"] = np.cos(2 * np.pi * result["Transaction_Hour"] / 24)
    return result.drop(columns=IDENTIFIERS + LEAKAGE_EXCLUDED + SUPPLIED_QUALITY_EXCLUDED + ["Timestamp", TARGET])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--random-state", type=int, default=42)
    args = parser.parse_args()

    files = list(Path("/opt/ml/processing/input").glob("*.csv"))
    if not files:
        raise FileNotFoundError("No dataset CSV supplied.")
    data = pd.read_csv(files[0])
    required = {
        "Transaction_ID", "User_ID", "Timestamp", "Risk_Score", "Is_Weekend",
        "Failed_Transaction_Count_7d", "Fraud_Label",
    }
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if data.isna().sum().sum() > 0:
        raise ValueError("Dataset contains missing values; review before pipeline execution.")
    if data.duplicated().sum() > 0:
        raise ValueError("Dataset contains duplicate rows; review before pipeline execution.")

    timestamp = pd.to_datetime(data["Timestamp"], errors="raise")
    derived_weekend = (timestamp.dt.dayofweek >= 5).astype(int)
    weekend_mismatch = int((derived_weekend != data["Is_Weekend"]).sum())
    suspected_rule = ((data["Risk_Score"] > 0.85) | (data["Failed_Transaction_Count_7d"] >= 4)).astype(int)
    rule_match_rate = float((suspected_rule == data[TARGET]).mean())
    print(f"Explicit rule match rate: {rule_match_rate:.8f}")
    print(f"Is_Weekend mismatch rows: {weekend_mismatch}")
    print("Risk_Score policy: EXCLUDED")

    X = prepare_features(data)
    y = data[TARGET].astype(int)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=args.random_state
    )
    X_train, X_validation, y_train, y_validation = train_test_split(
        X_train_val, y_train_val, test_size=0.25,
        stratify=y_train_val, random_state=args.random_state
    )

    output = Path("/opt/ml/processing/output")
    output.mkdir(parents=True, exist_ok=True)
    for name, features, labels in [
        ("train", X_train, y_train),
        ("validation", X_validation, y_validation),
        ("test", X_test, y_test),
    ]:
        frame = features.copy(); frame[TARGET] = labels.to_numpy()
        frame.to_csv(output / f"{name}.csv", index=False)
        print(f"{name}: rows={len(frame)}, fraud_rate={frame[TARGET].mean():.8f}")

    audit = {
        "rows": len(data), "duplicate_rows": 0, "missing_values": 0,
        "weekend_mismatch_rows": weekend_mismatch,
        "explicit_rule_match_rate": rule_match_rate,
        "risk_score_excluded": True,
        "split_strategy": "stratified_random_60_20_20",
    }
    (output / "pipeline_data_audit.json").write_text(json.dumps(audit, indent=2))


if __name__ == "__main__":
    main()
'''

train_source = r'''
import argparse
import base64
import json
import os
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def decode(value):
    return json.loads(base64.urlsafe_b64decode(value.encode("ascii")).decode("utf-8"))


def make_classifier(algorithm, params):
    params = dict(params)
    if algorithm == "LogisticRegression":
        return LogisticRegression(**params), True
    if algorithm == "XGBoost":
        from xgboost import XGBClassifier
        params["n_jobs"] = -1
        return XGBClassifier(**params), False
    raise ValueError(f"Unsupported algorithm: {algorithm}")


def evaluate(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "test_accuracy": float(accuracy_score(y_true, predictions)),
        "test_balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "test_precision": float(precision_score(y_true, predictions, zero_division=0)),
        "test_recall": float(recall_score(y_true, predictions, zero_division=0)),
        "test_f1": float(f1_score(y_true, predictions, zero_division=0)),
        "test_fpr": float(fp / (fp + tn) if fp + tn else 0.0),
        "test_alert_rate": float(predictions.mean()),
        "test_auc_roc": float(roc_auc_score(y_true, probabilities)),
        "test_auc_pr": float(average_precision_score(y_true, probabilities)),
        "test_mcc": float(matthews_corrcoef(y_true, predictions)),
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--algorithm", required=True)
    parser.add_argument("--model-family", required=True)
    parser.add_argument("--model-params-b64", required=True)
    parser.add_argument("--schema-b64", required=True)
    parser.add_argument("--decision-threshold", type=float, required=True)
    parser.add_argument("--source-run-id", required=True)
    args = parser.parse_args()

    params, schema = decode(args.model_params_b64), decode(args.schema_b64)
    train = pd.read_csv("/opt/ml/input/data/processed/train.csv")
    test = pd.read_csv("/opt/ml/input/data/processed/test.csv")
    X_train, y_train = train.drop(columns=["Fraud_Label"]), train["Fraud_Label"].astype(int)
    X_test, y_test = test.drop(columns=["Fraud_Label"]), test["Fraud_Label"].astype(int)
    if X_train.columns.tolist() != schema["model_feature_columns"]:
        raise ValueError("Processed training schema does not match Notebook 02 champion schema.")
    if "Risk_Score" in X_train.columns:
        raise RuntimeError("Risk_Score must never enter training.")

    classifier, scale = make_classifier(args.algorithm, params)
    preprocessor = ColumnTransformer([
        ("numeric", StandardScaler() if scale else "passthrough", schema["numeric_columns"]),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), schema["categorical_columns"]),
    ])
    model = Pipeline([("preprocess", preprocessor), ("classifier", classifier)])
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    metrics = evaluate(y_test, probabilities, args.decision_threshold)
    for name, value in metrics.items():
        print(f"{name}: {value:.8f}")

    model_dir = Path(os.environ["SM_MODEL_DIR"]); model_dir.mkdir(parents=True, exist_ok=True)
    bundle = {
        "model": model,
        "decision_threshold": args.decision_threshold,
        "algorithm": args.algorithm,
        "model_family": args.model_family,
        "source_mlflow_run_id": args.source_run_id,
        "raw_inference_columns": schema["raw_inference_columns"],
        "model_feature_columns": schema["model_feature_columns"],
        "risk_score_excluded": True,
    }
    joblib.dump(bundle, model_dir / "model_bundle.joblib")
    (model_dir / "evaluation.json").write_text(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()
'''

inference_source = r'''
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def model_fn(model_dir):
    bundle = joblib.load(Path(model_dir) / "model_bundle.joblib")
    if not bundle.get("risk_score_excluded"):
        raise RuntimeError("Unsafe model bundle: Risk_Score policy missing.")
    return bundle


def input_fn(request_body, content_type):
    if content_type != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")
    payload = json.loads(request_body)
    if isinstance(payload, dict):
        payload = [payload]
    if not isinstance(payload, list) or not payload:
        raise ValueError("Expected one JSON object or a non-empty list.")
    frame = pd.DataFrame(payload)
    forbidden = {"Risk_Score", "Fraud_Label", "Transaction_ID", "User_ID", "Is_Weekend"}
    supplied_forbidden = forbidden & set(frame.columns)
    if supplied_forbidden:
        raise ValueError(f"Forbidden input fields supplied: {sorted(supplied_forbidden)}")
    return frame


def prepare_features(frame, bundle):
    missing = [c for c in bundle["raw_inference_columns"] if c not in frame.columns]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")
    result = frame[bundle["raw_inference_columns"]].copy()
    timestamp = pd.to_datetime(result["Timestamp"], errors="raise")
    result["Transaction_Hour"] = timestamp.dt.hour
    result["Transaction_DayOfWeek"] = timestamp.dt.dayofweek
    result["Transaction_Month"] = timestamp.dt.month
    result["Derived_Is_Weekend"] = (timestamp.dt.dayofweek >= 5).astype(int)
    result["Transaction_Hour_Sin"] = np.sin(2 * np.pi * result["Transaction_Hour"] / 24)
    result["Transaction_Hour_Cos"] = np.cos(2 * np.pi * result["Transaction_Hour"] / 24)
    result = result.drop(columns=["Timestamp"])
    return result[bundle["model_feature_columns"]]


def predict_fn(data, bundle):
    features = prepare_features(data, bundle)
    probabilities = bundle["model"].predict_proba(features)[:, 1]
    threshold = float(bundle["decision_threshold"])
    predictions = (probabilities >= threshold).astype(int)
    return [{
        "prediction": int(prediction),
        "label": "Fraud" if prediction else "Non-fraud",
        "fraud_probability": float(probability),
        "decision_threshold": threshold,
        "algorithm": bundle["algorithm"],
    } for prediction, probability in zip(predictions, probabilities)]


def output_fn(prediction, accept):
    if accept not in ("application/json", "*/*"):
        raise ValueError(f"Unsupported accept type: {accept}")
    return json.dumps(prediction), "application/json"
'''

(SOURCE_DIR / "preprocess.py").write_text(preprocess_source, encoding="utf-8")
(SOURCE_DIR / "train.py").write_text(train_source, encoding="utf-8")
(SOURCE_DIR / "inference.py").write_text(inference_source, encoding="utf-8")
(SOURCE_DIR / "requirements.txt").write_text("xgboost>=2,<4\njoblib>=1.3\n", encoding="utf-8")
print("Generated:", [path.name for path in SOURCE_DIR.iterdir()])

Generated: ['.ipynb_checkpoints', 'preprocess.py', 'train.py', 'inference.py', 'requirements.txt', '_repack_model.py', '_repack_script_launcher.sh']


## 4. Upload Versioned Pipeline Inputs (dataset) and Source Artifacts

In [5]:
if not RAW_DATA_LOCAL.exists():
    raise FileNotFoundError(
        f"Missing {RAW_DATA_LOCAL}. Rename fraud_dataset(1).csv to fraud_dataset.csv and place it in data/."
    )
s3.upload_file(str(RAW_DATA_LOCAL), BUCKET, f"{PREFIX}/data/raw/fraud_dataset.csv")
source_files = [
    SOURCE_DIR / "preprocess.py",
    SOURCE_DIR / "train.py",
    SOURCE_DIR / "inference.py",
    SOURCE_DIR / "requirements.txt",
]
missing_source_files = [str(path) for path in source_files if not path.is_file()]
if missing_source_files:
    raise FileNotFoundError(
        f"Missing generated pipeline source files: {missing_source_files}. Run Section 3 first."
    )
for path in source_files:
    s3.upload_file(str(path), BUCKET, f"{PREFIX}/pipeline_src/{RAW_DATASET_VERSION}/{path.name}")
s3.upload_file(str(CHAMPION_FILE), BUCKET, f"{PREFIX}/pipeline_inputs/{CHAMPION_FILE.name}")
print("Upload complete.")

Upload complete.


## 5. Define and Register the Processing, Training and Staging Pipeline
Training and registration are allowed even when the source quality gate is false. Registration is evidence that a model artifact exists; it is **not** production approval. Every package created here remains `PendingManualApproval` and is treated as `Staging`.


In [6]:
from sagemaker.model import Model
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

pipeline_session = PipelineSession()
p_algorithm = ParameterString("Algorithm", default_value=CHAMPION_ALGORITHM)
p_family = ParameterString("ModelFamily", default_value=CHAMPION_MODEL_FAMILY)
p_params = ParameterString("ModelParamsB64", default_value=MODEL_PARAMS_B64)
p_schema = ParameterString("SchemaB64", default_value=SCHEMA_B64)
p_threshold = ParameterFloat("DecisionThreshold", default_value=DECISION_THRESHOLD)
p_min_recall = ParameterFloat("MinimumTestRecall", default_value=MIN_TEST_RECALL)
p_max_fpr = ParameterFloat("MaximumTestFPR", default_value=MAX_FALSE_POSITIVE_RATE)

processor = SKLearnProcessor(
    framework_version="1.2-1", instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-process",
)
step_process = ProcessingStep(
    name="AuditAndPreprocessData", processor=processor,
    inputs=[ProcessingInput(source=RAW_DATA_S3_URI, destination="/opt/ml/processing/input")],
    outputs=[ProcessingOutput(
        output_name="processed", source="/opt/ml/processing/output",
        destination=f"{PIPELINE_ROOT}/processed",
    )],
    code=str(SOURCE_DIR / "preprocess.py"),
    job_arguments=["--random-state", str(RANDOM_STATE)],
)

estimator = SKLearn(
    entry_point="train.py", source_dir=str(SOURCE_DIR), framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE, instance_count=1, role=role,
    sagemaker_session=pipeline_session,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    hyperparameters={
        "algorithm": p_algorithm, "model-family": p_family,
        "model-params-b64": p_params, "schema-b64": p_schema,
        "decision-threshold": p_threshold, "source-run-id": champion["best_run_id"],
    },
    metric_definitions=[
        {"Name": name, "Regex": f"{name}: ([0-9.]+)"}
        for name in [
            "test_accuracy", "test_balanced_accuracy", "test_precision", "test_recall",
            "test_f1", "test_fpr", "test_alert_rate", "test_auc_roc", "test_auc_pr", "test_mcc",
        ]
    ],
    tags=[
        {"Key": "Course", "Value": COURSE}, {"Key": "Semester", "Value": SEMESTER},
        {"Key": "TeamId", "Value": TEAM_ID}, {"Key": "StudentId", "Value": STUDENT_ID},
        {"Key": "DatasetVersion", "Value": RAW_DATASET_VERSION},
        {"Key": "RiskScoreExcluded", "Value": "true"},
    ],
)
processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri
step_train = TrainingStep(
    name="TrainLeakageSafeChampion", estimator=estimator,
    inputs={"processed": sagemaker.inputs.TrainingInput(s3_data=processed_uri, content_type="text/csv")},
)

model = Model(
    image_uri=estimator.training_image_uri(REGION),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role, entry_point="inference.py", source_dir=str(SOURCE_DIR),
    sagemaker_session=pipeline_session,
)
step_register = ModelStep(
    name="RegisterStagingCandidate",
    step_args=model.register(
        content_types=["application/json"], response_types=["application/json"],
        inference_instances=["ml.m5.large"], transform_instances=["ml.m5.large"],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status="PendingManualApproval",
        description=(
            f"{CHAMPION_MODEL_FAMILY}/{CHAMPION_ALGORITHM}; dataset={RAW_DATASET_VERSION}; "
            f"Risk_Score excluded; source MLflow run={champion['best_run_id']}; "
            f"lifecycle=Staging; source_deployment_ready={SOURCE_DEPLOYMENT_READY}"
        ),
    ),
)
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[p_algorithm, p_family, p_params, p_schema, p_threshold, p_min_recall, p_max_fpr],
    steps=[step_process, step_train, step_register],
    sagemaker_session=pipeline_session,
)
pipeline.upsert(role_arn=role)
print("Pipeline upserted:", PIPELINE_NAME)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/mast

Pipeline upserted: iti113-team07-credit-card-fraud-detection


## 6. Execute and Monitor the SageMaker Pipeline

In [7]:
execution = pipeline.start()
print("Execution ARN:", execution.arn)
previous = {}
while True:
    status = execution.describe()["PipelineExecutionStatus"]
    steps = execution.list_steps()
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])
    for step in steps:
        if previous.get(step["StepName"]) != step["StepStatus"]:
            print(f"{step['StepName']:<30} {step['StepStatus']}")
            previous[step["StepName"]] = step["StepStatus"]
    if status in {"Succeeded", "Failed", "Stopped"}:
        print("Pipeline status:", status); break
    time.sleep(30)
if status != "Succeeded":
    raise RuntimeError(f"Pipeline ended with status {status}.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team07-credit-card-fraud-detection/execution/91i26nm1lok9
AuditAndPreprocessData         Executing
TrainLeakageSafeChampion       Executing
AuditAndPreprocessData         Succeeded
RegisterStagingCandidate-RepackModel-0 Executing
TrainLeakageSafeChampion       Succeeded
RegisterStagingCandidate-RegisterModel Succeeded
RegisterStagingCandidate-RepackModel-0 Succeeded
Pipeline status: Succeeded


## 7. Evaluate Quality Gates and Record Pipeline Traceability

This section evaluates the captured pipeline metrics without blocking registration. A package that fails either gate remains a staging candidate with `DeploymentReady=false` and `PendingManualApproval`.

In [8]:
def get_steps(execution):
    response = execution.list_steps()
    return response if isinstance(response, list) else response.get("PipelineExecutionSteps", [])

def find_model_package_arn(value):
    if isinstance(value, str) and ":model-package/" in value:
        return value
    if isinstance(value, dict):
        for child in value.values():
            found = find_model_package_arn(child)
            if found: return found
    if isinstance(value, list):
        for child in value:
            found = find_model_package_arn(child)
            if found: return found
    return None

steps = get_steps(execution)
train_step = next(step for step in steps if step["StepName"] == "TrainLeakageSafeChampion")
training_job_arn = train_step["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]
training_job = sm.describe_training_job(TrainingJobName=training_job_name)
metric_names = {
    "test_accuracy", "test_balanced_accuracy", "test_precision", "test_recall",
    "test_f1", "test_fpr", "test_alert_rate", "test_auc_roc", "test_auc_pr", "test_mcc",
}
captured_metrics = {
    item["MetricName"]: float(item["Value"])
    for item in training_job.get("FinalMetricDataList", [])
    if item["MetricName"] in metric_names
}
if metric_names - set(captured_metrics):
    raise RuntimeError(f"Missing captured metrics: {sorted(metric_names - set(captured_metrics))}")

model_package_arn = find_model_package_arn(steps)
if not model_package_arn:
    raise RuntimeError("Pipeline completed without producing the expected staging package.")
model_artifact_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]

pipeline_recall_passed = captured_metrics["test_recall"] >= MIN_TEST_RECALL
pipeline_fpr_passed = captured_metrics["test_fpr"] <= MAX_FALSE_POSITIVE_RATE
# Source readiness is retained so a failed Notebook 02 result cannot silently become
# production-ready simply because the pipeline reran the same model and split.
deployment_ready = bool(
    SOURCE_DEPLOYMENT_READY and pipeline_recall_passed and pipeline_fpr_passed
)
quality_gate_status = "Passed" if deployment_ready else "Failed"

# SageMaker supports AddTags on a Model Package Group, not on an individual
# numbered Model Package version. Keep version-specific gate results in MLflow
# and the run summary below; tag the group only with stable group metadata.
model_package_group_arn = sm.describe_model_package_group(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP
)["ModelPackageGroupArn"]
sm.add_tags(
    ResourceArn=model_package_group_arn,
    Tags=[
        {"Key": "Course", "Value": COURSE},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "StudentId", "Value": STUDENT_ID},
        {"Key": "ProjectName", "Value": PROJECT_NAME},
        {"Key": "DatasetVersion", "Value": RAW_DATASET_VERSION},
        {"Key": "RiskScoreExcluded", "Value": "true"},
    ],
)

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_pipeline_{CHAMPION_ALGORITHM}_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": COURSE, "semester": SEMESTER, "team_id": TEAM_ID,
        "student_id": STUDENT_ID, "project_name": PROJECT_NAME,
        "dataset_version": RAW_DATASET_VERSION, "model_family": CHAMPION_MODEL_FAMILY,
        "algorithm": CHAMPION_ALGORITHM, "risk_score_excluded": "true",
        "feature_policy": champion["feature_policy"]["name"],
        "source_experiment_run_id": champion["best_run_id"],
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "model_package_arn": model_package_arn,
        "model_artifact_s3_uri": model_artifact_uri,
        "lifecycle_stage": "Staging",
        "model_approval_status": "PendingManualApproval",
        "deployment_ready": str(deployment_ready).lower(),
        "quality_gate_status": quality_gate_status,
    })
    mlflow.log_params({
        **{f"champion_{key}": value for key, value in CHAMPION_PARAMS.items()},
        "decision_threshold": DECISION_THRESHOLD,
        "minimum_test_recall": MIN_TEST_RECALL,
        "maximum_test_fpr": MAX_FALSE_POSITIVE_RATE,
    })
    mlflow.log_metrics(captured_metrics)
    summary = {
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "model_artifact_s3_uri": model_artifact_uri,
        "model_package_arn": model_package_arn,
        "source_mlflow_run_id": champion["best_run_id"],
        "dataset_version": RAW_DATASET_VERSION,
        "risk_score_excluded": True,
        "algorithm": CHAMPION_ALGORITHM,
        "lifecycle_stage": "Staging",
        "model_approval_status": "PendingManualApproval",
        "source_deployment_ready": SOURCE_DEPLOYMENT_READY,
        "pipeline_recall_passed": pipeline_recall_passed,
        "pipeline_fpr_passed": pipeline_fpr_passed,
        "deployment_ready": deployment_ready,
        "quality_gate_status": quality_gate_status,
        "metrics": captured_metrics,
    }
    summary_path = Path("sagemaker_pipeline_run_summary.json")
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    mlflow.log_artifact(str(summary_path), artifact_path="sagemaker_pipeline")
    PIPELINE_MLFLOW_RUN_ID = run.info.run_id

print("Model package:", model_package_arn)
print("Metrics:", captured_metrics)
print("Lifecycle stage: Staging")
print("Model approval status: PendingManualApproval")
print("Recall gate passed:", pipeline_recall_passed)
print("FPR gate passed:", pipeline_fpr_passed)
print("Deployment ready:", deployment_ready)

🏃 View run team07_s701_pipeline_XGBoost_1787680133 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/aee744b3443e48d596f2908c85f48aa5
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
Model package: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/13
Metrics: {'test_balanced_accuracy': 0.7984293699264526, 'test_fpr': 0.03992927819490433, 'test_auc_roc': 0.8134356737136841, 'test_alert_rate': 0.23170000314712524, 'test_accuracy': 0.8561999797821045, 'test_precision': 0.8830384016036987, 'test_recall': 0.6367880702018738, 'test_auc_pr': 0.8092902302742004, 'test_mcc': 0.6605982780456543, 'test_f1': 0.7399638295173645}
Lifecycle stage: Staging
Model approval status: PendingManualApproval
Recall gate passed: True
FPR gate passed: True
Deployment ready: True


## 8. Verify Staging Safety Controls Before Manual Promotion

The following cells deliberately do **not** approve the model or create an endpoint. Production approval must be a separate manual governance decision after the quality gate passes on acceptable evidence.


In [9]:
package_details = sm.describe_model_package(ModelPackageName=model_package_arn)
approval_status = package_details["ModelApprovalStatus"]

if ALLOW_PRODUCTION_APPROVAL:
    raise RuntimeError(
        "Automatic production approval is prohibited in this staging notebook."
    )
if approval_status == "Approved":
    raise RuntimeError(
        "Safety check failed: the newly registered staging package is already Approved."
    )

print("Production approval skipped.")
print("Model package status:", approval_status)
print("Lifecycle stage: Staging")
print("Deployment ready:", deployment_ready)


Production approval skipped.
Model package status: PendingManualApproval
Lifecycle stage: Staging
Deployment ready: True


In [10]:
predictor = None
endpoint_created = False

if ALLOW_ENDPOINT_DEPLOYMENT:
    raise RuntimeError(
        "Automatic endpoint deployment is prohibited in this staging notebook."
    )

print("Endpoint deployment skipped.")
print("Reason: model is registered for staging evaluation only.")
print("Deployment ready:", deployment_ready)


Endpoint deployment skipped.
Reason: model is registered for staging evaluation only.
Deployment ready: True


## 9. Confirm the Inference Contract Before Deployment and print staging pipeline summary

In [11]:
# The inference script and JSON contract were created earlier, but live endpoint
# testing is intentionally skipped because this notebook does not deploy an endpoint.
if predictor is None:
    print("Live inference test skipped: no endpoint was created.")
    print("The raw JSON contract remains available in pipeline_src/inference.py.")

print("=" * 72)
print("STAGING PIPELINE COMPLETE")
print("=" * 72)
print("Dataset              :", RAW_DATASET_VERSION)
print("Champion             :", CHAMPION_MODEL_FAMILY, "/", CHAMPION_ALGORITHM)
print("Risk_Score excluded  : True")
print("Source MLflow run    :", champion["best_run_id"])
print("Pipeline MLflow run  :", PIPELINE_MLFLOW_RUN_ID)
print("Model package        :", model_package_arn)
print("Lifecycle stage      : Staging")
print("Approval status      :", approval_status)
print("Test recall/FPR      :", captured_metrics["test_recall"], "/", captured_metrics["test_fpr"])
print("Deployment ready     :", deployment_ready)
print("Endpoint created     :", endpoint_created)
print("Production action    : Manual review required; no automatic approval/deployment")

Live inference test skipped: no endpoint was created.
The raw JSON contract remains available in pipeline_src/inference.py.
STAGING PIPELINE COMPLETE
Dataset              : fraud_dataset_50k_v1
Champion             : Model_B / XGBoost
Risk_Score excluded  : True
Source MLflow run    : 21825ef39b7c41bdadc838f378ae6c2f
Pipeline MLflow run  : aee744b3443e48d596f2908c85f48aa5
Model package        : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/13
Lifecycle stage      : Staging
Approval status      : PendingManualApproval
Test recall/FPR      : 0.6367880702018738 / 0.03992927819490433
Deployment ready     : True
Endpoint created     : False
Production action    : Manual review required; no automatic approval/deployment


## 10. List Registered Model Package Versions

In [12]:
import boto3

REGION = "ap-southeast-1"
MODEL_PACKAGE_GROUP = "team07-credit-card-fraud-detection"

sm = boto3.client("sagemaker", region_name=REGION)

response = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
)

for package in response["ModelPackageSummaryList"]:
    print(
        "Version:", package["ModelPackageVersion"],
        "| Status:", package["ModelPackageStatus"],
        "| Approval:", package["ModelApprovalStatus"],
        "\nARN:", package["ModelPackageArn"],
        "\n",
    )

Version: 13 | Status: Completed | Approval: PendingManualApproval 
ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/13 

Version: 12 | Status: Completed | Approval: Approved 
ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/12 

Version: 11 | Status: Completed | Approval: Approved 
ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/11 

Version: 10 | Status: Completed | Approval: Approved 
ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/10 

Version: 9 | Status: Completed | Approval: Approved 
ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/9 

Version: 8 | Status: Completed | Approval: PendingManualApproval 
ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-credit-card-fraud-detection/8 

Version: 7 | Status: Com

## 11. Review and Manually Approve the Staging Model Package

In [13]:
import boto3

sm = boto3.client("sagemaker",region_name=REGION,)

MODEL_PACKAGE_ARN_TO_APPROVE = model_package_arn

# Retrieve the model package produced by this pipeline execution.
details = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN_TO_APPROVE
)
MODEL_PACKAGE_VERSION = str(details["ModelPackageVersion"])

print("Model group:", details["ModelPackageGroupName"])
print("Model version:", details["ModelPackageVersion"])
print("Package status:", details["ModelPackageStatus"])
print("Current approval:", details["ModelApprovalStatus"])
print("Description:", details.get("ModelPackageDescription", ""))

# Quality result produced by Section 7.
if globals().get("deployment_ready") is not True:
    raise RuntimeError(
        "Approval blocked: run Notebook 3 Section 7 and confirm "
        "Deployment ready is True."
    )

if details["ModelPackageStatus"] != "Completed":
    raise RuntimeError("Approval blocked: model package is not Completed.")

if details["ModelApprovalStatus"] == "Approved":
    print(f"Version {MODEL_PACKAGE_VERSION} is already approved; no update required.")
else:
    confirmation = input(
        f"Type APPROVE to approve {TEAM_ID} model version "
        f"{MODEL_PACKAGE_VERSION}: "
    ).strip()

    if confirmation != "APPROVE":
        raise RuntimeError("Approval cancelled.")

    sm.update_model_package(
        ModelPackageArn=MODEL_PACKAGE_ARN_TO_APPROVE,
        ModelApprovalStatus="Approved",
        ApprovalDescription=(
            "Manually approved after Notebook 3 quality gates passed: "
            "test recall >= 0.60, test FPR <= 0.05, "
            "Risk_Score excluded."
        ),
    )

    updated = sm.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN_TO_APPROVE)
    print("Updated approval status:",updated["ModelApprovalStatus"],)

Model group: team07-credit-card-fraud-detection
Model version: 13
Package status: Completed
Current approval: PendingManualApproval
Description: Model_B/XGBoost; dataset=fraud_dataset_50k_v1; Risk_Score excluded; source MLflow run=21825ef39b7c41bdadc838f378ae6c2f; lifecycle=Staging; source_deployment_ready=True


Type APPROVE to approve team07 model version 13:  APPROVE


Updated approval status: Approved


## 12. Create and Deploy a Temporary SageMaker Endpoint

In [14]:
import time
import boto3
import sagemaker

sm = boto3.client("sagemaker", region_name=REGION)
role = sagemaker.get_execution_role()

MODEL_PACKAGE_ARN = MODEL_PACKAGE_ARN_TO_APPROVE

# Recheck approval before deployment.
package = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)
MODEL_PACKAGE_VERSION = str(package["ModelPackageVersion"])

if package["ModelPackageStatus"] != "Completed":
    raise RuntimeError("Deployment blocked: package is not Completed.")

if package["ModelApprovalStatus"] != "Approved":
    raise RuntimeError("Deployment blocked: package is not Approved.")

timestamp = int(time.time())

RESOURCE_PREFIX = f"{TEAM_ID}-fraud-v{MODEL_PACKAGE_VERSION}"
MODEL_NAME = f"{RESOURCE_PREFIX}-{timestamp}"
ENDPOINT_CONFIG_NAME = f"{RESOURCE_PREFIX}-config-{timestamp}"
ENDPOINT_NAME = f"{RESOURCE_PREFIX}-endpoint-{timestamp}"

sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=role,
    Containers=[
        {
            "ModelPackageName": MODEL_PACKAGE_ARN,
        }
    ],
    Tags=[
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "ModelPackageVersion", "Value": MODEL_PACKAGE_VERSION},
        {"Key": "Purpose", "Value": "TemporaryTesting"},
    ],
)

sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.m5.large",
            "InitialVariantWeight": 1.0,
        }
    ],
    Tags=[
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "Purpose", "Value": "TemporaryTesting"},
    ],
)

sm.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    Tags=[
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "Purpose", "Value": "TemporaryTesting"},
    ],
)

print("Endpoint creation started.")
print("Endpoint name:", ENDPOINT_NAME)
print("Waiting for InService status...")

waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=ENDPOINT_NAME,
    WaiterConfig={
        "Delay": 30,
        "MaxAttempts": 60,
    },
)

endpoint_details = sm.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)

print("Endpoint status:", endpoint_details["EndpointStatus"])

Endpoint creation started.
Endpoint name: team07-fraud-v13-endpoint-1787680145
Waiting for InService status...
Endpoint status: InService


## 13. Validate the Live Endpoint with a Sample Prediction

In [15]:
import json
from pathlib import Path

import boto3
import pandas as pd

DATASET_PATH = RAW_DATA_LOCAL

sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

# Section 13 creates and stores the exact endpoint name in ENDPOINT_NAME.
if "ENDPOINT_NAME" not in globals():
    raise RuntimeError(
        "ENDPOINT_NAME is unavailable. Run Section 13 first."
    )

endpoint = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)

if endpoint["EndpointStatus"] != "InService":
    raise RuntimeError(
        f"Endpoint is not ready: {endpoint['EndpointStatus']}"
    )

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

if not CHAMPION_FILE.exists():
    raise FileNotFoundError(
        f"Champion file not found: {CHAMPION_FILE}"
    )

# Read the exact inference-column contract saved by Notebook 2.
champion = json.loads(
    CHAMPION_FILE.read_text(encoding="utf-8")
)
raw_inference_columns = champion["raw_inference_columns"]

test_data = pd.read_csv(DATASET_PATH)

missing_columns = [
    column
    for column in raw_inference_columns
    if column not in test_data.columns
]

if missing_columns:
    raise RuntimeError(
        f"Dataset is missing inference fields: {missing_columns}"
    )

# Convert the first transaction to ordinary JSON-compatible values.
test_payload = json.loads(
    test_data.loc[[0], raw_inference_columns].to_json(
        orient="records",
        date_format="iso",
    )
)[0]

print("Endpoint:", ENDPOINT_NAME)
print("Endpoint status:", endpoint["EndpointStatus"])
print("\nRequest:")
print(json.dumps(test_payload, indent=2))

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(test_payload).encode("utf-8"),
)

prediction = json.loads(
    response["Body"].read().decode("utf-8")
)

print("\nPrediction:")
print(json.dumps(prediction, indent=2))

Endpoint: team07-fraud-v13-endpoint-1787680145
Endpoint status: InService

Request:
{
  "Transaction_Amount": 39.79,
  "Transaction_Type": "POS",
  "Timestamp": "2023-08-14 19:30:00",
  "Account_Balance": 93213.17,
  "Device_Type": "Laptop",
  "Location": "Sydney",
  "Merchant_Category": "Travel",
  "IP_Address_Flag": 0,
  "Previous_Fraudulent_Activity": 0,
  "Daily_Transaction_Count": 7,
  "Avg_Transaction_Amount_7d": 437.63,
  "Failed_Transaction_Count_7d": 3,
  "Card_Type": "Amex",
  "Card_Age": 65,
  "Transaction_Distance": 883.17,
  "Authentication_Method": "Biometric"
}

Prediction:
[
  {
    "prediction": 0,
    "label": "Non-fraud",
    "fraud_probability": 0.15825606882572174,
    "decision_threshold": 0.19469599425792694,
    "algorithm": "XGBoost"
  }
]


## 14. Delete the Temporary Endpoint Resources

In [16]:
#sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
#print("Endpoint deletion started:", ENDPOINT_NAME)
#sm.get_waiter("endpoint_deleted").wait(EndpointName=ENDPOINT_NAME, WaiterConfig={
#        "Delay": 30,"MaxAttempts": 60 })
#sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
#sm.delete_model(ModelName=MODEL_NAME)
#print("Temporary endpoint, configuration and model removed.")